# SAM2 Video Predictor — スターターノートブック

Week 1 の最小 E2E 貫通用。Colab T4 GPU で動作することを前提に書かれています。

**目的**: 任意の車載クリップに対しクリック1点で SAM2 video predictor を動かし、アノテーション付き MP4 を出力する。

**所要時間目安**: 初回セットアップ込みで約 20〜30 分。

## 進め方

1. ランタイム → ランタイムのタイプを変更 → ハードウェアアクセラレータ = T4 GPU
2. 上から順にセルを実行
3. 「自前クリップ差し替え」セクションで自分の動画に切り替え

## 1. 環境チェック

In [ ]:
!nvidia-smi
import torch
print('torch:', torch.__version__)
print('cuda available:', torch.cuda.is_available())
print('device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')

## 2. SAM2 のインストール

公式リポジトリを clone し、editable インストール。

In [ ]:
%cd /content
!git clone https://github.com/facebookresearch/sam2.git
%cd /content/sam2
!pip install -e . -q
!pip install opencv-python matplotlib -q

## 3. チェックポイントのダウンロード

SAM2.1 Hiera-Large を使用。Tiny/Small/Base+ も同梱スクリプトで取得可能。

In [ ]:
%cd /content/sam2/checkpoints
!./download_ckpts.sh
!ls -lh /content/sam2/checkpoints/

## 4. インポートと device 設定

In [ ]:
import os, sys
import numpy as np
import torch
import matplotlib.pyplot as plt
from PIL import Image
import cv2

sys.path.insert(0, '/content/sam2')
from sam2.build_sam import build_sam2_video_predictor

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
torch.autocast(device_type=device.type, dtype=torch.bfloat16).__enter__()
if device.type == 'cuda':
    torch.backends.cuda.matmul.allow_tf32 = True
    torch.backends.cudnn.allow_tf32 = True
print('using device:', device)

## 5. predictor のロード

In [ ]:
sam2_checkpoint = '/content/sam2/checkpoints/sam2.1_hiera_large.pt'
model_cfg = 'configs/sam2.1/sam2.1_hiera_l.yaml'
predictor = build_sam2_video_predictor(model_cfg, sam2_checkpoint, device=device)
print('predictor ready')

## 6. 動画 → JPEG フレーム展開

SAM2 video predictor はフレームディレクトリを入力に取るため、先に展開する。

**初回確認用**: 公式リポジトリ同梱のサンプル動画を使用。  
**自前クリップ差し替え**: 下のセルで `VIDEO_PATH` を書き換える。

In [ ]:
# === 設定: ここを書き換えると自前クリップに差し替えられます ===
# 公式サンプル(寝室) は notebooks/videos/bedroom にフレームが既にある
USE_OFFICIAL_SAMPLE = True

if USE_OFFICIAL_SAMPLE:
    frame_dir = '/content/sam2/notebooks/videos/bedroom'
    print('using official sample frames at:', frame_dir)
else:
    # 自前クリップを使う場合はここにアップロード後のパスを書く
    VIDEO_PATH = '/content/your_clip.mp4'  # 例: 5〜15秒の車載クリップ
    frame_dir = '/content/frames'
    os.makedirs(frame_dir, exist_ok=True)
    # ffmpeg で 0-padded JPEG に展開 (SAM2 が要求するフォーマット)
    !ffmpeg -y -i "{VIDEO_PATH}" -q:v 2 -start_number 0 "{frame_dir}/%05d.jpg"
    print('frames extracted to:', frame_dir)

frame_names = sorted([f for f in os.listdir(frame_dir) if f.lower().endswith(('.jpg', '.jpeg'))])
print(f'total frames: {len(frame_names)}')

## 7. 推論状態の初期化

In [ ]:
inference_state = predictor.init_state(video_path=frame_dir)
print('inference state initialized')

## 8. 追跡開始フレームを表示しクリック点を決める

最初のフレームを表示し、目で見て追跡対象の (x, y) を決める。

In [ ]:
ann_frame_idx = 0
img = Image.open(os.path.join(frame_dir, frame_names[ann_frame_idx]))
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.title(f'frame {ann_frame_idx} — 追跡したい物体の中心 (x, y) を読み取る')
plt.axis('on')
plt.show()
print('画像サイズ:', img.size)

In [ ]:
# === 設定: クリック点 (x, y) ===
# 上のグラフの軸を見て、追跡したい物体の中心座標を入れる
click_xy = (350, 250)  # 例: 必要に応じて変更

points = np.array([[click_xy[0], click_xy[1]]], dtype=np.float32)
labels = np.array([1], dtype=np.int32)  # 1 = positive (追跡対象)

frame_idx, out_obj_ids, out_mask_logits = predictor.add_new_points_or_box(
    inference_state=inference_state,
    frame_idx=ann_frame_idx,
    obj_id=1,
    points=points,
    labels=labels,
)

# クリック点と最初のマスクを可視化
mask = (out_mask_logits[0] > 0.0).cpu().numpy()
if mask.ndim == 3:
    mask = mask[0]
plt.figure(figsize=(10, 6))
plt.imshow(img)
plt.imshow(mask, alpha=0.5, cmap='Reds')
plt.scatter([click_xy[0]], [click_xy[1]], c='lime', s=120, marker='*', edgecolors='black')
plt.title('frame 0 — click + initial mask')
plt.axis('off')
plt.show()

## 9. 全フレームへ伝播

In [ ]:
video_segments = {}
for out_frame_idx, out_obj_ids, out_mask_logits in predictor.propagate_in_video(inference_state):
    video_segments[out_frame_idx] = {
        oid: (out_mask_logits[i] > 0.0).cpu().numpy()
        for i, oid in enumerate(out_obj_ids)
    }
print('propagation done. frames covered:', len(video_segments))

## 10. 結果を MP4 として書き出す

In [ ]:
OUTPUT_DIR = '/content/output'
OVERLAY_DIR = os.path.join(OUTPUT_DIR, 'overlay_frames')
os.makedirs(OVERLAY_DIR, exist_ok=True)

for idx, fname in enumerate(frame_names):
    img_bgr = cv2.imread(os.path.join(frame_dir, fname))
    if idx in video_segments:
        for oid, m in video_segments[idx].items():
            mask_arr = m
            if mask_arr.ndim == 3:
                mask_arr = mask_arr[0]
            color = np.array([0, 0, 255], dtype=np.uint8)  # BGR red
            overlay = img_bgr.copy()
            overlay[mask_arr > 0] = (overlay[mask_arr > 0] * 0.5 + color * 0.5).astype(np.uint8)
            img_bgr = overlay
    cv2.imwrite(os.path.join(OVERLAY_DIR, f'{idx:05d}.jpg'), img_bgr)

out_mp4 = os.path.join(OUTPUT_DIR, 'tracked.mp4')
!ffmpeg -y -framerate 24 -i "{OVERLAY_DIR}/%05d.jpg" -c:v libx264 -pix_fmt yuv420p "{out_mp4}"
print('saved:', out_mp4)

In [ ]:
# Colab 上で確認
from IPython.display import Video
Video(out_mp4, embed=True, width=720)

## 11. 自前クリップでの差し替え手順

1. 左サイドバーのファイルから `your_clip.mp4` を `/content/` にアップロード
2. セル6の `USE_OFFICIAL_SAMPLE = False` に変更、`VIDEO_PATH` を実ファイルパスに
3. セル6から再実行
4. セル8の `click_xy` を、自分のクリップ向けに調整
